**Movie Recommendations**

This notebook demonstrates different movie recommendation systems using the MovieLens 100K dataset.

**Importing dependencies**

In [1]:
import urllib.request
import zipfile
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

**Data Loading and Preprocessing**

In [2]:
# 1. Download and Extract MovieLens 100K
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = "ml-100k.zip"
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")

# 2. Load User Ratings (u.data)
# u.data columns: user_id, item_id (movie_id), rating, timestamp
ratings = pd.read_csv(
    'ml-100k/u.data',
    sep='\t',
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)

# 3. Load Movies Metadata (u.item)
genre_cols = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
item_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + genre_cols

movies = pd.read_csv(
    'ml-100k/u.item',
    sep='|',
    names=item_cols,
    encoding='latin-1'
)

print(f"Loaded {len(ratings)} ratings and {len(movies)} movies.")

Loaded 100000 ratings and 1682 movies.


**Content-Based Recommendation**

- Genre Matrix Creation: It creates a feature matrix where each row represents a movie and columns represent genres (binary values indicating presence or absence of a genre).

- Cosine Similarity: Calculates the cosine similarity between all pairs of movies based on their genre features. This results in movie_sim_df, a similarity matrix where movie_sim_df[i][j] is the genre similarity between movie i and movie j.

- get_content_recommendations function: Takes a movie_title as input and returns top_n movies that are most similar in terms of genre. It finds the target movie's ID, retrieves its similarity scores from movie_sim_df, sorts them, and returns the top similar movies (excluding the movie itself).

- Test: Demonstrates recommendations for 'Toy Story'.

In [3]:
# Create a Genre Matrix (Movie Features)
genre_features = movies[genre_cols]

# Compute Cosine Similarity between every pair of movies
movie_similarity_matrix = cosine_similarity(genre_features)
movie_sim_df = pd.DataFrame(
    movie_similarity_matrix,
    index=movies['movie_id'],
    columns=movies['movie_id']
)

def get_content_recommendations(movie_title, top_n=5):
    # Find movie_id for given title
    match = movies[movies['title'].str.contains(movie_title, case=False, na=False)]
    if match.empty:
        return f"Movie '{movie_title}' not found."

    movie_id = match.iloc[0]['movie_id']
    target_title = match.iloc[0]['title']

    # Sort movies by genre similarity score
    similar_scores = movie_sim_df[movie_id].sort_values(ascending=False)

    # Exclude the query movie itself
    top_movie_ids = similar_scores.iloc[1:top_n+1].index

    rec_movies = movies[movies['movie_id'].isin(top_movie_ids)][['title']]
    print(f"Top {top_n} recommendations similar to '{target_title}':")
    return rec_movies

# Test Content-Based Model
get_content_recommendations("Toy Story", top_n=5)

Top 5 recommendations similar to 'Toy Story (1995)':


,title
93,Home Alone (1990)
94,Aladdin (1992)
421,Aladdin and the King of Thieves (1996)
1218,"Goofy Movie, A (1995)"
1469,Gumby: The Movie (1995)


**Collaborative Filtering (Item-Based) Recommendation**

- User-Item Pivot Table: Creates user_item_matrix where rows are user_ids, columns are movie_ids, and values are ratings.
- Rating Normalization: Normalizes the ratings by subtracting each user's average rating. This helps account for differences in how users rate movies (e.g., some users generally give higher ratings than others).
- Item-Item Similarity: Computes cosine similarity between movies based on their normalized rating vectors across users. This forms item_collab_df, representing how similarly movies are rated by users.
- get_collaborative_recommendations function: For a given user_id, it identifies movies the user has rated highly (rating >= 4). It then scores unrated movies by summing the collaborative similarities to these highly-rated movies, weighted by the user's rating for the liked movie. Finally, it recommends the top_n unrated movies with the highest scores.
- Test: Demonstrates recommendations for User ID 1.

In [4]:
# 1. Create User-Item Pivot Table (Rows: Users, Columns: Movies)
user_item_matrix = ratings.pivot(index='user_id', columns='movie_id', values='rating')

# Normalize ratings by subtracting each user's average rating (handles strict vs lenient raters)
user_item_matrix_norm = user_item_matrix.apply(lambda row: row - row.mean(), axis=1).fillna(0)

# 2. Compute Item-Item Similarity matrix based on rating vectors
item_collaborative_sim = cosine_similarity(user_item_matrix_norm.T)
item_collab_df = pd.DataFrame(
    item_collaborative_sim,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

def get_collaborative_recommendations(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return f"User {user_id} not found."

    # Get movies the user has rated highly (rating >= 4)
    user_ratings = user_item_matrix.loc[user_id].dropna()
    liked_movies = user_ratings[user_ratings >= 4].index

    if len(liked_movies) == 0:
        return "User hasn't rated enough movies highly."

    # Score candidate movies based on collaborative similarity to liked movies
    scores = pd.Series(dtype=float)
    for movie_id in liked_movies:
        similar_items = item_collab_df[movie_id] * user_ratings[movie_id]
        scores = scores.add(similar_items, fill_value=0)

    # Filter out movies the user has already rated
    unrated_scores = scores.drop(labels=user_ratings.index, errors='ignore')
    top_movie_ids = unrated_scores.sort_values(ascending=False).head(top_n).index

    rec_movies = movies[movies['movie_id'].isin(top_movie_ids)][['movie_id', 'title']]
    print(f"Top {top_n} Collaborative Filtering Recommendations for User {user_id}:")
    return rec_movies

# Test Collaborative Model for User ID 1
get_collaborative_recommendations(user_id=1, top_n=5)

Top 5 Collaborative Filtering Recommendations for User 1:


,movie_id,title
317,318,Schindler's List (1993)
356,357,One Flew Over the Cuckoo's Nest (1975)
482,483,Casablanca (1942)
602,603,Rear Window (1954)
650,651,Glory (1989)


**Hybrid Recommendation (Item-Based Collaborative + Content-Based)**

- get_hybrid_recommendations function: Combines content-based scores (genre similarity to a target_movie_title) and collaborative filtering scores (from item_collab_df based on user_id's liked movies).
- Normalization: Both content and collaborative scores are normalized to a [0, 1] range using MinMaxScaler to make them comparable.
- Weighted Combination: A weighted sum (alpha * content_norm) + ((1 - alpha) * collab_norm) is used to produce a final hybrid score. alpha controls the balance between content and collaborative filtering.
- Filtering: Removes movies the user has already rated and the target_movie_id itself.
- Cold Start Handling: If a user is not found, it defaults to pure content-based filtering (sets alpha = 1.0).
- Test: Demonstrates hybrid recommendations for User ID 1 with 'Toy Story' as the content context.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def get_hybrid_recommendations(user_id, target_movie_title, alpha=0.5, top_n=5):
    """
    Generates hybrid recommendations by combining genre content similarity
    and item-based collaborative filtering scores.
    """
    # 1. Locate the target query movie
    match = movies[movies['title'].str.contains(target_movie_title, case=False, na=False)]
    if match.empty:
        return f"Movie '{target_movie_title}' not found."

    target_movie_id = match.iloc[0]['movie_id']

    # 2. Extract Raw Content Scores (Cosine similarity based on genres)
    content_scores = movie_sim_df[target_movie_id].copy()

    # 3. Extract Raw Collaborative Scores for the User
    if user_id not in user_item_matrix.index:
        # Fallback to pure content-based if user is unknown (Cold Start)
        print(f"User {user_id} not found. Defaulting to pure Content-Based Filtering.")
        alpha = 1.0
        collab_scores = pd.Series(0, index=movies['movie_id'])
    else:
        user_ratings = user_item_matrix.loc[user_id].dropna()
        liked_movies = user_ratings[user_ratings >= 4].index

        collab_scores = pd.Series(0.0, index=item_collab_df.index)
        for m_id in liked_movies:
            collab_scores = collab_scores.add(item_collab_df[m_id] * user_ratings[m_id], fill_value=0)

    # 4. Normalize Both Score Vectors to [0, 1] range
    scaler = MinMaxScaler()

    content_norm = pd.Series(
        scaler.fit_transform(content_scores.values.reshape(-1, 1)).flatten(),
        index=content_scores.index
    )

    collab_norm = pd.Series(
        scaler.fit_transform(collab_scores.values.reshape(-1, 1)).flatten(),
        index=collab_scores.index
    )

    # 5. Calculate Weighted Hybrid Score
    hybrid_scores = (alpha * content_norm) + ((1 - alpha) * collab_norm)

    # 6. Filter out movies already rated by the user & query movie itself
    if user_id in user_item_matrix.index:
        already_rated = user_item_matrix.loc[user_id].dropna().index
        hybrid_scores = hybrid_scores.drop(labels=already_rated, errors='ignore')

    hybrid_scores = hybrid_scores.drop(labels=[target_movie_id], errors='ignore')

    # 7. Extract Top N Recommendations
    top_movie_ids = hybrid_scores.sort_values(ascending=False).head(top_n).index

    results = movies[movies['movie_id'].isin(top_movie_ids)][['movie_id', 'title']]
    results['hybrid_score'] = hybrid_scores.loc[top_movie_ids].values

    print(f"Top {top_n} Hybrid Recommendations (User: {user_id}, Seed Movie: '{target_movie_title}', Alpha: {alpha}):")
    return results.sort_values(by='hybrid_score', ascending=False)

# Test Hybrid System for User ID 1 with 'Toy Story' as content context
get_hybrid_recommendations(user_id=1, target_movie_title="Toy Story", alpha=0.4, top_n=5)

Top 5 Hybrid Recommendations (User: 1, Seed Movie: 'Toy Story', Alpha: 0.4):


,movie_id,title,hybrid_score
403,404,Pinocchio (1940),0.718168
407,408,"Close Shave, A (1995)",0.675461
419,420,Alice in Wonderland (1951),0.642801
432,433,Heathers (1989),0.624749
1006,1007,Waiting for Guffman (1996),0.622035


**SVD Matrix Factorization with Surprise Library**

- Installation: Installs the scikit-surprise library.
- Data Preparation: Formats the ratings DataFrame into a Surprise Dataset object.
- Train-Test Split: Splits the data into training and testing sets for model evaluation.
- SVD Model Training: Initializes and trains an SVD (Singular Value Decomposition) model, a popular matrix factorization technique for collaborative filtering, on the training set.
- Evaluation: Evaluates the SVD model's performance on the test set using RMSE (Root Mean Squared Error).

In [6]:
# 1. Install Scikit-Surprise
!pip install scikit-surprise

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# 2. Format Data for Surprise (User, Item, Rating)
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['user_id', 'movie_id', 'rating']], reader)

# 3. Split into Train & Test sets to verify model accuracy
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# 4. Train SVD Matrix Factorization
svd_model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

# Evaluate performance
predictions = svd_model.test(testset)
print("SVD Model RMSE:")
accuracy.rmse(predictions)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.8 MB/s eta 0:00:00
SVD Model RMSE:
RMSE: 0.9348


np.float64(0.934761145254825)

**Hybrid Recommendation (SVD Collaborative + Content-Based)**

- get_svd_hybrid_recommendations function: This is another hybrid approach, this time combining SVD's predicted ratings with content-based similarity.
- SVD Predictions: Uses the trained svd_model to predict ratings for all movies for a given user_id.
- Content Scores: Retrieves content similarity scores for a target_movie_title (if provided), similar to the pure content-based model.
- Normalization and Weighted Combination: Both SVD predicted ratings and content scores are normalized and combined using alpha.
- Dynamic Alpha: If no target_movie_title is provided, alpha defaults to 1.0 (pure SVD recommendation).
- Filtering: Filters out movies the user has already rated.
- Test: Demonstrates both pure SVD recommendations and SVD + Content hybrid recommendations.

In [7]:
from sklearn.preprocessing import MinMaxScaler

def get_svd_hybrid_recommendations(user_id, target_movie_title=None, alpha=0.6, top_n=5):
    """
    Hybrid Recommender combining SVD Matrix Factorization with Content-Based Similarity.
    - alpha = 1.0 -> 100% SVD Collaborative Filtering
    - alpha = 0.0 -> 100% Genre Content Filtering
    """
    all_movie_ids = movies['movie_id'].unique()

    # 1. Predict ratings for ALL movies for the given user using SVD
    svd_predictions = [svd_model.predict(user_id, m_id).est for m_id in all_movie_ids]
    svd_series = pd.Series(svd_predictions, index=all_movie_ids)

    # 2. Get Content Similarity Scores (if a seed movie title is provided)
    if target_movie_title:
        match = movies[movies['title'].str.contains(target_movie_title, case=False, na=False)]
        if not match.empty:
            target_movie_id = match.iloc[0]['movie_id']
            content_series = movie_sim_df[target_movie_id]
        else:
            print(f"Movie '{target_movie_title}' not found. Relying solely on SVD.")
            content_series = pd.Series(0.0, index=all_movie_ids)
    else:
        content_series = pd.Series(0.0, index=all_movie_ids)

    # 3. Normalize Both Scores to [0, 1] Range
    scaler = MinMaxScaler()

    collab_norm = pd.Series(
        scaler.fit_transform(svd_series.values.reshape(-1, 1)).flatten(),
        index=svd_series.index
    )

    content_norm = pd.Series(
        scaler.fit_transform(content_series.values.reshape(-1, 1)).flatten(),
        index=content_series.index
    ) if target_movie_title else pd.Series(0.0, index=all_movie_ids)

    # 4. Calculate Weighted Score
    # If no target movie provided, default alpha to 1.0 (pure SVD)
    effective_alpha = alpha if target_movie_title else 1.0
    hybrid_scores = (effective_alpha * collab_norm) + ((1 - effective_alpha) * content_norm)

    # 5. Filter out movies the user has already rated
    if user_id in user_item_matrix.index:
        already_rated = user_item_matrix.loc[user_id].dropna().index
        hybrid_scores = hybrid_scores.drop(labels=already_rated, errors='ignore')
        svd_series = svd_series.drop(labels=already_rated, errors='ignore')

    # 6. Extract Top-N Recommendations
    top_movie_ids = hybrid_scores.sort_values(ascending=False).head(top_n).index

    results = movies[movies['movie_id'].isin(top_movie_ids)][['movie_id', 'title']].copy()
    results['predicted_rating'] = svd_series.loc[top_movie_ids].values
    results['hybrid_score'] = hybrid_scores.loc[top_movie_ids].values

    print(f"--- Top {top_n} SVD Hybrid Recommendations for User {user_id} ---")
    return results.sort_values(by='hybrid_score', ascending=False)

# Test pure SVD recommendation for User ID 1
get_svd_hybrid_recommendations(user_id=1, alpha=1.0, top_n=5)

# Test SVD + Content Hybrid (User 1 context + 'Toy Story' genre preference)
get_svd_hybrid_recommendations(user_id=1, target_movie_title="Toy Story", alpha=0.6, top_n=5)

--- Top 5 SVD Hybrid Recommendations for User 1 ---
--- Top 5 SVD Hybrid Recommendations for User 1 ---


,movie_id,title,predicted_rating,hybrid_score
403,404,Pinocchio (1940),4.701608,0.823586
407,408,"Close Shave, A (1995)",3.917171,0.734966
431,432,Fantasia (1940),4.173447,0.723566
587,588,Beauty and the Beast (1991),4.102498,0.710130
968,969,Winnie the Pooh and the Blustery Day (1968),3.784315,0.709807


**Top-$K$ ranking metrics**

Top-$K$ ranking metrics evaluate what users actually interact with: Are the items at the top of the recommendation list relevant to the user?To calculate Precision@K, Recall@K, and F1-score@K, we define a rating threshold (typically $\ge 3.5$ or $\ge 4.0$) to classify items as Relevant or Irrelevant.Key Metric Definitions

Precision@K: What fraction of the recommended top-$K$ items are actually relevant?$$\text{Precision}@K = \frac{\vert{}\text{Relevant Items} \cap \text{Recommended Top-}K\vert{}}{K}
$$Recall@K: What fraction of all relevant items in the test set were successfully captured in the top-$K$?$$\text{Recall}@K = \frac{\vert{}\text{Relevant Items} \cap \text{Recommended Top-}K\vert{}}{\vert{}\text{All Relevant Items}\vert{}}$$F1-Score@K: The harmonic mean of Precision@K and Recall@K.$$\text{F1}@K = 2 \cdot \frac{\text{Precision}@K \cdot \text{Recall}@K}{\text{Precision}@K + \text{Recall}@K}$$AUC (Area Under ROC Curve): The probability that a randomly chosen relevant item is ranked higher by the model than a randomly chosen irrelevant item.

In [8]:
from collections import defaultdict
from surprise.model_selection import KFold

def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """
    Return Precision@K and Recall@K for each user in predictions.
    """
    # 1. Map predictions to each user: user_id -> [(est_rating, true_rating), ...]
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()

    for uid, user_ratings in user_est_true.items():
        # Sort user predictions by estimated rating in descending order
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Number of relevant items in true ratings
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Number of recommended items in top-k
        n_rec_k = min(k, len(user_ratings))

        # Number of relevant items recommended in top-k
        n_rel_and_rec_k = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in user_ratings[:k]
        )

        # Precision@K: Portion of recommended items that are relevant
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

        # Recall@K: Portion of relevant items that are recommended
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return precisions, recalls


# --- Evaluate SVD Model across 5 Folds ---
kf = KFold(n_splits=5, random_state=42)

precisions_list = []
recalls_list = []

for trainset, testset in kf.split(data):
    svd_model.fit(trainset)
    predictions = svd_model.test(testset)

    precisions, recalls = precision_recall_at_k(predictions, k=10, threshold=3.5)

    # Average over all users in the fold
    precisions_list.append(sum(prec_val for prec_val in precisions.values()) / len(precisions))
    recalls_list.append(sum(rec_val for rec_val in recalls.values()) / len(recalls))

# Mean Precision & Recall across 5 folds
mean_precision = sum(precisions_list) / len(precisions_list)
mean_recall = sum(recalls_list) / len(recalls_list)
mean_f1 = 2 * (mean_precision * mean_recall) / (mean_precision + mean_recall) if (mean_precision + mean_recall) > 0 else 0

print(f"--- Evaluation at K=10 (Threshold = 3.5) ---")
print(f"Mean Precision@10 : {mean_precision:.4f}")
print(f"Mean Recall@10    : {mean_recall:.4f}")
print(f"Mean F1-Score@10  : {mean_f1:.4f}")

--- Evaluation at K=10 (Threshold = 3.5) ---
Mean Precision@10 : 0.5726
Mean Recall@10    : 0.5389
Mean F1-Score@10  : 0.5552


Inference

Top-K Ranking Metrics (at K=10, Threshold = 3.5)
These metrics focus on how well the model recommends relevant items in the top K positions.

Mean Precision@10: 0.5726

Explanation: This means that, on average, about 57.26% of the movies recommended in the top 10 for any given user were actually relevant to that user (based on a rating of 3.5 or higher).

Mean Recall@10: 0.5389

Explanation: This indicates that, on average, the model successfully captured about 53.89% of all the relevant movies for a user within its top 10 recommendations. In other words, for every 10 relevant movies a user has, the model managed to recommend approximately 5-6 of them in its top 10 list.

Mean F1-Score@10: 0.5552

Explanation: The F1-Score is the harmonic mean of Precision and Recall. A score of 0.5552 suggests a reasonable balance between recommending relevant items and covering a good portion of all relevant items. It's a single metric that balances both concerns.


**Evaluating ROC-AUC for Ranking Performance**

To compute AUC, we convert ratings into binary ground truth ($1$ if rating $\ge 3.5$, else $0$) and evaluate the global capability of the estimated ratings using scikit-learn

In [9]:
from sklearn.metrics import roc_auc_score

# Extract true ratings and estimated ratings
y_true_binary = [1 if true_r >= 3.5 else 0 for (_, _, true_r, _, _) in predictions]
y_score = [est for (_, _, _, est, _) in predictions]

auc_score = roc_auc_score(y_true_binary, y_score)
print(f"Global ROC-AUC Score: {auc_score:.4f}")

Global ROC-AUC Score: 0.7829


Inference

ROC-AUC Score

Global ROC-AUC Score: 0.7829

Explanation: The Area Under the Receiver Operating Characteristic Curve (AUC) measures the model's ability to distinguish between relevant (rating >= 3.5) and irrelevant (rating < 3.5) items. An AUC of 0.7829 (or 78.29%) means that there's a 78.29% chance that the model will rank a randomly chosen relevant item higher than a randomly chosen irrelevant item. This is considered a good score, indicating that the SVD model has a strong capability to correctly identify and prioritize items that users would likely consider relevant.

**LLM for movie recommendation**

In [11]:
!pip install -qU chromadb sentence-transformers groq| tail -n 1

In [14]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


In [15]:
import os
from google.colab import userdata
# 1. Set Groq API Key
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [19]:
from groq import Groq
groq_client = Groq()

In [20]:
import os
import pandas as pd
import numpy as np
import chromadb
from chromadb.utils import embedding_functions

# ==========================================
# STEP 1: Feature Engineering & Preprocessing
# ==========================================

def extract_genres(row):
    return ", ".join([genre for genre in genre_cols if row[genre] == 1])

movies['genres_text'] = movies.apply(extract_genres, axis=1)

# 2. Aggregate Collaborative Rating Signals
user_stats = ratings.groupby('movie_id').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

# 3. Merge Metadata with Collaborative Signals
movies_enriched = movies.merge(user_stats, on='movie_id', how='left')
movies_enriched['avg_rating'] = movies_enriched['avg_rating'].fillna(0).round(1)
movies_enriched['rating_count'] = movies_enriched['rating_count'].fillna(0).astype(int)

# 4. Create Rich Document Representations for Sentence Transformer
def build_movie_profile(row):
    release_year = str(row['release_date']).split('-')[-1] if pd.notnull(row['release_date']) else "Unknown"
    return (
        f"Title: {row['title']} | Year: {release_year} | "
        f"Genres: {row['genres_text']} | "
        f"Average Rating: {row['avg_rating']}/5 from {row['rating_count']} user reviews."
    )

movies_enriched['text_profile'] = movies_enriched.apply(build_movie_profile, axis=1)

# ==========================================
# STEP 2: ChromaDB & Sentence Transformer Indexing
# ==========================================

# Initialize ChromaDB persistent client
chroma_client = chromadb.PersistentClient(path="./chroma_movielens_db")

# Use SentenceTransformers embedding model
st_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create or get collection
collection = chroma_client.get_or_create_collection(
    name="movielens_100k",
    embedding_function=st_ef
)

# Index movies into Vector DB
ids = [str(mid) for mid in movies_enriched['movie_id'].tolist()]
documents = movies_enriched['text_profile'].tolist()
metadatas = [
    {
        "title": row['title'],
        "genres": row['genres_text'],
        "avg_rating": float(row['avg_rating']),
        "rating_count": int(row['rating_count'])
    }
    for _, row in movies_enriched.iterrows()
]

# Upsert items (batched implicitly by Chroma)
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f"Successfully indexed {collection.count()} movies into ChromaDB.")

# ==========================================
# STEP 3: Vector Search Retrieval Function
# ==========================================

def retrieve_candidate_movies(query_text, n_results=5):
    """
    Retrieves top N candidate movies based on semantic similarity.
    """
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results
    )

    candidates = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        candidates.append({
            "profile": doc,
            "title": meta['title'],
            "genres": meta['genres'],
            "avg_rating": meta['avg_rating']
        })
    return candidates

# ==========================================
# STEP 4: LLM Explanation with Groq (Llama 3)
# ==========================================

def generate_llm_recommendations(user_preference, top_k=5):
    """
    Retrieves candidates from ChromaDB and passes them to Groq for generation.
    """
    # 1. Vector Search
    candidates = retrieve_candidate_movies(user_preference, n_results=top_k)

    # 2. Construct Prompt Context
    candidates_context = "\n".join(
        [f"- {c['profile']}" for c in candidates]
    )

    prompt = f"""You are an expert movie recommendation engine.
A user provided the following preference query:
"{user_preference}"

Here are the top candidate movies retrieved from our Vector Database based on genres, semantics, and community ratings:
{candidates_context}

Task:
1. Explain why each movie fits the user's preference query.
2. Present the recommendations in a clean, numbered list.
3. Highlight the standout feature (genre blend or rating confidence) for each choice.
"""

    # 3. Request LLM Inference from Groq API
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful and concise movie recommendation assistant."},
            {"role": "user", "content": prompt}
        ],
        model="openai/gpt-oss-20b",
        temperature=0.3
    )

    return chat_completion.choices[0].message.content

# ==========================================
# STEP 5: Run Recommendation Query
# ==========================================

query = "I love dark sci-fi action thrillers with high audience ratings."
recommendation_response = generate_llm_recommendations(query, top_k=5)

print(f"User Query: {query}\n")
print(recommendation_response)

Successfully indexed 1682 movies into ChromaDB.
User Query: I love dark sci-fi action thrillers with high audience ratings.

**Why each title matches (or partially matches) your “dark sci‑fi action thriller” taste**

| # | Title | Why it fits | Stand‑out feature |
|---|-------|-------------|-------------------|
| 1 | **Deep Rising** (1998) | • Action‑heavy, sci‑fi setting on a submarine.<br>• Dark, gritty tone with a relentless chase. | **Genre blend** – action + sci‑fi + horror gives a “dark” edge, though the low rating (2.4/5) signals limited audience approval. |
| 2 | **Assignment, The** (1997) | • Pure thriller with tense pacing and a mysterious plot.<br>• Dark atmosphere and suspenseful twists. | **Thriller core** – the film delivers the “thriller” element, but it lacks sci‑fi, so it’s a partial match. |
| 3 | **Dark City** (1998) | • Classic film‑noir meets sci‑fi mystery.<br>• Dark, moody visuals and a high‑stakes thriller narrative. | **Genre blend** – film‑noir + sci‑fi + thri